In [14]:
! uv pip install pinecone pinecone-text pinecone-notebooks --reinstall
! uv pip uninstall pinecone-client -y

Using Python 3.13.5 environment at: /Users/niharikareddy/Desktop/Assignments/Udemy/ML_Projects/GIT_HUB/Langchain_Chat_Bot/.venv
Resolved 24 packages in 265ms                                        
Prepared 24 packages in 11ms                                             
Uninstalled 24 packages in 658ms
Installed 24 packages in 152ms                              
 ~ certifi==2026.2.25
 ~ charset-normalizer==3.4.4
 ~ click==8.3.1
 ~ idna==3.11
 ~ joblib==1.5.3
 ~ mmh3==4.1.0
 ~ nltk==3.9.3
 ~ numpy==2.4.2
 ~ orjson==3.11.7
 ~ packaging==24.2
 ~ pinecone==8.1.0
 ~ pinecone-notebooks==0.1.1
 ~ pinecone-plugin-assistant==3.0.2
 ~ pinecone-plugin-interface==0.0.7
 ~ pinecone-text==0.11.0
 ~ python-dateutil==2.9.0.post0
 ~ python-dotenv==1.2.2
 ~ regex==2026.2.28
 ~ requests==2.32.5
 ~ six==1.17.0
 ~ tqdm==4.67.3
 ~ types-requests==2.32.4.20260107
 ~ typing-extensions==4.15.0
 ~ urllib3==2.6.3
error: unexpected argument '-y' found

  tip: to pass '-y' as a value, use '-- -y'

Usage: uv pip u

In [15]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
# API KEY
api_key = os.getenv('PINECONE_API_KEY')

In [17]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [19]:
from pinecone import Pinecone,ServerlessSpec
index_name = "hybrid-search-langchain-pinecone"
# Initilize the pinecone client
pc = Pinecone(api_key=api_key)

# create the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension = 1536,#dimension of dense vector
        metric = 'dotproduct', # specifically for sparse value supported only for dotproduct
        spec=ServerlessSpec(cloud='aws',region='us-east-1')
    )


In [20]:
index =pc.Index(index_name)
index

/Users/niharikareddy/Desktop/Assignments/Udemy/ML_Projects/GIT_HUB/Langchain_Chat_Bot/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
# Vector Embedding and Sparse Matrix
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
embeddings

/Users/niharikareddy/Desktop/Assignments/Udemy/ML_Projects/GIT_HUB/Langchain_Chat_Bot/.venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/niharikareddy/Desktop/Assignments/Udemy/ML_Projects/GIT_HUB/Langchain_Chat_Bot/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
# Vector Embedding and Sparse Matrix
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
embeddings

ValidationError: 1 validation error for HuggingFaceEmbeddings
model
  extra fields not permitted (type=value_error.extra)

In [ ]:
import pinecone, os
print(list(pinecone.__path__))




['/Users/niharikareddy/Desktop/Assignments/Udemy/ML_Projects/GIT_HUB/Langchain_Chat_Bot/.venv/lib/python3.13/site-packages/pinecone']


In [ ]:
import pinecone, os
print(list(pinecone.__path__))




['/Users/niharikareddy/Desktop/Assignments/Udemy/ML_Projects/GIT_HUB/Langchain_Chat_Bot/.venv/lib/python3.13/site-packages/pinecone']


In [ ]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder().default()
bm25_encoder

In [ ]:
sentences=[
    'In 2023, I visited Paris',
    'In 2022, I visited New York',
    'In 2021, I visited New Orleans',
]
# TFIDF values on these sentence
bm25_encoder.fit(sentences)

# store the values to a json file
bm25_encoder.dump('bm25_values.json')

100%|██████████| 3/3 [00:00<00:00, 4329.98it/s]


In [ ]:
retriever = PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index)

In [ ]:
retriever

PineconeHybridSearchRetriever(embeddings=OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x130f3fb10>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x1321e0410>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version='', openai_api_base=None, openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=set(), disallowed_special='all', chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x130dcf100>, index=<pinecone.db_data.index.Index object at 0x115a69090>)

In [ ]:
retriever.add_texts(
    [
    'In 2023, I visited Paris',
    'In 2022, I visited New York',
    'In 2021, I visited New Orleans',
    ]
)

  0%|          | 0/1 [00:00<?, ?it/s]


PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Thu, 05 Mar 2026 00:59:00 GMT', 'Content-Type': 'application/json', 'Content-Length': '103', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '85', 'x-envoy-upstream-service-time': '43', 'x-pinecone-response-duration-ms': '90', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Vector dimension 1536 does not match the dimension of the index 384","details":[]}
